# 03 — Inference Profiling

Characterise the deployed model's inference performance:
- Latency vs sequence length
- Tokens-per-second across batch sizes
- Sampling parameter sensitivity
- Quantisation quality trade-off
- Side-by-side generation comparison (base vs fine-tuned)

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import statistics
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from src.inference.engine import LlamaCppEngine, SamplingConfig
from src.inference.sampler import (
    apply_temperature, apply_top_p, apply_min_p,
    effective_vocab_size, token_entropy,
)

sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams['figure.dpi'] = 120

GGUF_PATH = '../outputs/model_q4km.gguf'  # update to your model

In [ ]:
# Load model
engine = LlamaCppEngine(
    GGUF_PATH,
    n_ctx=4096,
    n_gpu_layers=-1,
    verbose=False,
)
print('Model loaded.')

In [ ]:
# Latency vs output length
PROMPT = 'Explain the transformer architecture in detail, covering attention, positional encoding, and training.'
max_token_configs = [64, 128, 256, 512]
n_runs = 3

latencies = {}
for max_tokens in max_token_configs:
    sampling = SamplingConfig(temperature=0.0, max_tokens=max_tokens, seed=42)
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = engine.generate(PROMPT, sampling)
        times.append(time.perf_counter() - t0)
    latencies[max_tokens] = statistics.mean(times)
    print(f'max_tokens={max_tokens:4d}  mean_latency={latencies[max_tokens]:.2f}s')

plt.figure(figsize=(8, 4))
plt.plot(list(latencies.keys()), list(latencies.values()), 'o-', color='steelblue', lw=2)
plt.xlabel('Max Output Tokens')
plt.ylabel('Latency (s)')
plt.title('Inference Latency vs Output Length')
plt.tight_layout()
plt.show()

In [ ]:
# Tokens-per-second measurement
sampling = SamplingConfig(temperature=0.0, max_tokens=256, seed=42)
bench = engine.benchmark(PROMPT, sampling, n_runs=5)

print(f'Time to first token (mean): {bench["mean_ttft_ms"]:.1f} ms')
print(f'Tokens per second (mean):   {bench["mean_tok_per_sec"]:.1f} tok/s')

In [ ]:
# Sampling parameter sensitivity — visualise effective vocab size
import torch
import torch.nn.functional as F

# Simulate a peaked distribution (like a confident model)
logits_peaked = torch.randn(32000)
logits_peaked[42] += 5.0  # one clearly preferred token

temperatures = [0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]
eff_sizes = []
entropies = []

for t in temperatures:
    scaled = apply_temperature(logits_peaked.clone(), t)
    eff_sizes.append(effective_vocab_size(scaled, p=0.95))
    entropies.append(token_entropy(scaled))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(temperatures, eff_sizes, 'o-', color='steelblue', lw=2)
axes[0].set_xlabel('Temperature')
axes[0].set_ylabel('Effective Vocab Size (p95)')
axes[0].set_title('How temperature broadens the distribution')

axes[1].plot(temperatures, entropies, 'o-', color='coral', lw=2)
axes[1].set_xlabel('Temperature')
axes[1].set_ylabel('Entropy (nats)')
axes[1].set_title('Distribution entropy vs temperature')

plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side generation comparison
test_prompts = [
    'What is the difference between LoRA and full fine-tuning?',
    'Write a Python function that implements binary search.',
    'Explain gradient descent in simple terms.',
]

sampling = SamplingConfig(temperature=0.7, top_p=0.9, max_tokens=200, seed=42)

print('=' * 70)
for prompt in test_prompts:
    print(f'\nPROMPT: {prompt}')
    print('-' * 50)
    response = engine.generate(prompt, sampling)
    print(response.strip())
    print('=' * 70)

In [ ]:
# Streaming demo
print('STREAMING DEMO\n' + '='*40)
sampling_stream = SamplingConfig(temperature=0.7, max_tokens=150)
prompt = 'Explain what quantisation means for neural networks.'
print(f'Prompt: {prompt}\n')
print('Response: ', end='')

for token in engine.stream(prompt, sampling_stream):
    print(token, end='', flush=True)
print()  # newline at end